# GBFS Feed Exploration\nInvestigate what station data is available from the Divvy real-time feeds.

In [1]:
import requests
import pandas as pd

# Discover all available feeds
gbfs = requests.get("https://gbfs.divvybikes.com/gbfs/gbfs.json").json()
feeds = {f["name"]: f["url"] for f in gbfs["data"]["en"]["feeds"]}
feeds

{'gbfs': 'https://gbfs.lyft.com/gbfs/1.1/chi/gbfs.json',
 'ebikes_at_stations': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/ebikes_at_stations.json',
 'system_information': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/system_information.json',
 'station_information': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/station_information.json',
 'station_status': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/station_status.json',
 'free_bike_status': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/free_bike_status.json',
 'system_hours': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/system_hours.json',
 'system_calendar': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/system_calendar.json',
 'system_regions': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/system_regions.json',
 'system_pricing_plans': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/system_pricing_plans.json',
 'system_alerts': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/system_alerts.json',
 'gbfs_versions': 'https://gbfs.lyft.com/gbfs/1.1/chi/en/gbfs_versions.json'}

## Station Information\nStatic metadata — name, location, capacity.

In [2]:
station_info = requests.get(feeds["station_information"]).json()
df_info = pd.DataFrame(station_info["data"]["stations"])
print(df_info.shape)
df_info.head()

(1930, 16)


,rental_uris,lon,eightd_station_services,capacity,short_name,external_id,station_type,name,electric_bike_surcharge_waiver,eightd_has_key_dispenser,lat,rental_methods,station_id,has_kiosk,region_id,address
0,"{'ios': 'https://chi.lft.to/lastmile_qr_scan',...",-87.70136,[],8,CHI02208,2186823631363753926,classic,Kedzie Ave & 104th St,False,False,41.70382,"[KEY, CREDITCARD, TRANSITCARD]",2186823631363753926,False,NaN,NaN
1,"{'ios': 'https://chi.lft.to/lastmile_qr_scan',...",-87.76124,[],7,CHI01932,1961472602929588766,classic,Long Ave & Diversey Ave,False,False,41.93104,"[KEY, CREDITCARD, TRANSITCARD]",1961472602929588766,False,NaN,NaN
2,"{'ios': 'https://chi.lft.to/lastmile_qr_scan',...",-87.78202,[],8,CHI01954,2007796988006173238,classic,Melvina Ave & Giddings Ave,False,False,41.96648,"[KEY, CREDITCARD, TRANSITCARD]",2007796988006173238,False,NaN,NaN
3,"{'ios': 'https://chi.lft.to/lastmile_qr_scan',...",-87.65774,[],16,CHI01970,2013067286975627866,classic,Loomis St & 95th St,False,False,41.72148,"[KEY, CREDITCARD, TRANSITCARD]",2013067286975627866,False,NaN,NaN
4,"{'ios': 'https://chi.lft.to/lastmile_qr_scan',...",-87.61439,[],15,CHI01973,1970688521031469268,classic,Martin Luther King Dr & 87th St,False,False,41.73677,"[KEY, CREDITCARD, TRANSITCARD]",1970688521031469268,False,NaN,NaN


## Check Ingested Ridership Data\nRead back January 2024 from S3 dev and inspect.

In [3]:
from dotenv import load_dotenv
from divvy_demand.infra.s3_client import S3Client

load_dotenv("../.env")

s3 = S3Client(env="dev")
df = s3.read_raw(2024, 1)

print(df.shape)
print(df.dtypes)
df.head()

(144873, 13)
ride_id                   str
rideable_type             str
started_at                str
ended_at                  str
start_station_name        str
start_station_id          str
end_station_name          str
end_station_id            str
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual             str
dtype: object


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,C1D650626C8C899A,electric_bike,2024-01-12 15:30:27,2024-01-12 15:37:59,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.903267,-87.634737,41.889177,-87.638506,member
1,EECD38BDB25BFCB0,electric_bike,2024-01-08 15:45:46,2024-01-08 15:52:59,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902937,-87.634440,41.889177,-87.638506,member
2,F4A9CE78061F17F7,electric_bike,2024-01-27 12:27:19,2024-01-27 12:35:19,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902951,-87.634470,41.889177,-87.638506,member
3,0A0D9E15EE50B171,classic_bike,2024-01-29 16:26:17,2024-01-29 16:56:06,Wells St & Randolph St,TA1305000030,Larrabee St & Webster Ave,13193,41.884295,-87.633963,41.921822,-87.644140,member
4,33FFC9805E3EFF9A,classic_bike,2024-01-31 05:43:23,2024-01-31 06:09:35,Lincoln Ave & Waveland Ave,13253,Kingsbury St & Kinzie St,KA1503000043,41.948797,-87.675278,41.889177,-87.638506,member


In [ ]:
df.isnull().sum()